# 第三讲：随机数生成与蒙特卡洛模拟

**学习目标**
- 掌握新的 Generator API，告别过时的 `np.random.seed()`
- 熟练使用 `uniform`、`normal`、`integers` 三大常用分布
- 理解随机种子与可复现性的底层逻辑
- 用蒙特卡洛方法解决两个经典问题：掷骰子实验 + 估算 π

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体（macOS 用 Heiti SC，Windows 用 SimHei）
plt.rcParams['font.sans-serif'] = ['Heiti SC', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"NumPy 版本: {np.__version__}")

---

## 3.1 Generator API —— 新一代随机数引擎

### 为什么要换？

老式写法 `np.random.seed(42)` 用的是全局状态，多线程下会互相干扰，且算法老旧（MT19937）。
从 NumPy 1.17 开始推荐使用 `Generator` + `PCG64` 引擎，更快、更安全、更灵活。

```
❌ 过时写法:  np.random.seed(42);  np.random.rand(3)
✅ 推荐写法:  rng = np.random.default_rng(42);  rng.random(3)
```

In [ ]:
# 创建一个随机数生成器实例（推荐做法）
rng = np.random.default_rng(seed=42)

# 生成 5 个 [0, 1) 均匀分布的随机数
print("rng.random(5):", rng.random(5))
print()

# 验证可复现性：用同样的 seed 再创建一个新的 rng，结果完全相同
rng2 = np.random.default_rng(seed=42)
print("rng2.random(5):", rng2.random(5))
print("完全相同?", np.array_equal(rng.random(5), rng2.random(5)))

### 关键理解：种子决定"起点"

随机数生成器本质是一个**确定性算法**——给定相同的种子，产生完全相同的随机数序列。
\n这恰恰是科学计算需要的：**实验可复现**。你可以把种子理解为"随机数序列的编号"，
同样的编号永远对应同样的序列。

In [ ]:
# 演示：不同种子产生不同序列，相同种子产生相同序列

seeds = [42, 123, 2024]

for s in seeds:
    rng = np.random.default_rng(seed=s)
    print(f"seed={s:>4} → 前 5 个随机数: {rng.random(5)}")

print()
# 再次用 seed=42，验证结果一致
rng = np.random.default_rng(seed=42)
print(f"seed=42  再次确认:   {rng.random(5)}")

---

## 3.2 uniform —— 均匀分布

最基础的分布。每个值在指定区间内"等可能"出现。

```python
rng.uniform(low=0.0, high=1.0, size=None)
```

- `low`：下界（含）
- `high`：上界（不含）
- `size`：输出的形状

In [ ]:
rng = np.random.default_rng(seed=42)

# 10 个 [0, 1) 的随机数
print("[0, 1):   ", rng.uniform(0, 1, 10))

# 5 个 [-5, 5) 的随机数
print("[-5, 5):  ", rng.uniform(-5, 5, 5))

# 3×4 矩阵，范围 [10, 20)
print("\n3×4 矩阵 [10, 20):")
print(rng.uniform(10, 20, size=(3, 4)))

### 可视化均匀分布

In [ ]:
# 生成 100000 个 [0, 1) 均匀分布随机数，画直方图
rng = np.random.default_rng(seed=123)
samples = rng.uniform(0, 1, 100_000)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(samples, bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='white')
ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='理论概率密度 = 1.0')
ax.set_title('均匀分布 U(0,1) — 100,000 个样本', fontsize=14)
ax.set_xlabel('值')
ax.set_ylabel('概率密度')
ax.legend()
plt.tight_layout()
plt.show()

---

## 3.3 normal —— 正态分布（高斯分布）

自然界和金融市场中最常见的分布。

```python
rng.normal(loc=0.0, scale=1.0, size=None)
```

- `loc`：均值 μ（分布的中心）
- `scale`：标准差 σ（分布的宽度）

In [ ]:
rng = np.random.default_rng(seed=42)

# 标准正态分布 N(0, 1)
print("标准正态 N(0,1):", rng.normal(0, 1, 5))

# 均值为 5，标准差为 2
print("N(5, 2):", rng.normal(5, 2, 5))

# 2×3 矩阵
print("\n2×3 矩阵 N(100, 15):")
print(rng.normal(100, 15, size=(2, 3)))

In [ ]:
# 可视化正态分布
rng = np.random.default_rng(seed=42)
samples = rng.normal(loc=0, scale=1, size=100_000)

fig, ax = plt.subplots(figsize=(10, 5)) #创建画布和坐标轴（绘图基础）
ax.hist(samples, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')

# 叠加理论正态曲线
x = np.linspace(-4, 4, 200) #- 4 到 4 之间生成200 个等间距的点
pdf = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi) #标准正态分布 N (0,1) 的概率密度函数
ax.plot(x, pdf, color='red', linewidth=2, label='理论 N(0,1) 曲线')

ax.set_title('标准正态分布 N(0,1) — 100,000 个样本', fontsize=14)
ax.set_xlabel('值')
ax.set_ylabel('概率密度')
ax.legend()
plt.tight_layout() #自动调整图表所有元素的位置，避免标题、坐标轴标签被画布边缘截断
plt.show()

---

## 3.4 integers —— 随机整数

模拟掷骰子、抽样等离散场景的核心工具。

```python
rng.integers(low, high=None, size=None, endpoint=False)
```

- `low`：下界（含）
- `high`：上界（**默认不含**，除非 `endpoint=True`）
- `endpoint`：是否包含 `high`

In [ ]:
rng = np.random.default_rng(seed=42)

# [0, 10) 的随机整数（不含 10）
print("[0, 10): ", rng.integers(0, 10, 10))

# [1, 7) → 模拟掷骰子：1 到 6
print("掷 5 次骰子:", rng.integers(1, 7, 5))

# 使用 endpoint=True 让上限也包含在内
print("[1, 6] 含两端:", rng.integers(1, 6, 10, endpoint=True))

# 3×4 矩阵，范围 [0, 100)
print("\n3×4 矩阵 [0, 100):")
print(rng.integers(0, 100, size=(3, 4)))

---

## 3.5 其他常用分布速查

| 方法 | 分布 | 用途 |
|------|------|------|
| `rng.uniform(low, high)` | 均匀分布 | 等概率随机 |
| `rng.normal(loc, scale)` | 正态分布 | 收益率建模 |
| `rng.integers(low, high)` | 随机整数 | 抽样、骰子 |
| `rng.exponential(scale)` | 指数分布 | 等待时间 |
| `rng.poisson(lam)` | 泊松分布 | 事件计数 |
| `rng.binomial(n, p)` | 二项分布 | 成功次数 |
| `rng.choice(array, size)` | 随机抽样 | 从数组中抽 |

---

## 3.6 蒙特卡洛实战 ①：模拟掷骰子 10,000 次

**问题：** 投掷一颗公平的六面骰子 10,000 次，统计各点数出现频率，验证是否接近 1/6。

In [ ]:
rng = np.random.default_rng(seed=42)
n_trials = 10_000

# 一次模拟 10,000 次掷骰子
dice_rolls = rng.integers(1, 7, size=n_trials)

print(f"前 20 次结果: {dice_rolls[:20]}")
print(f"平均值: {dice_rolls.mean():.3f} (理论值: 3.5)")
print(f"标准差: {dice_rolls.std():.3f} (理论值: 1.708)")

In [ ]:
# 统计每个点数出现的次数
faces, counts = np.unique(dice_rolls, return_counts=True)

print("点数频率分布:")
print("-" * 35)
for face, count in zip(faces, counts):
    freq = count / n_trials
    bar = '█' * int(freq * 200)
    print(f"  点数 {face}: {count:>5} 次  ({freq:.4f})  | 理论: 1/6 ≈ {1/6:.4f}  {bar}")

print("-" * 35)
print(f"  总计:     {counts.sum():>5} 次")

In [ ]:
# 绘制频率分布直方图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：频数柱状图
colors = ['#FF6B6B', '#FFA94D', '#FFD43B', '#69DB7C', '#4DABF7', '#DA77F2']
axes[0].bar(faces, counts, color=colors, edgecolor='white', linewidth=1.5)
axes[0].axhline(y=n_trials/6, color='black', linestyle='--', linewidth=1.5, label=f'理论值: {n_trials/6:.0f}')
axes[0].set_title(f'掷骰子 {n_trials:,} 次 — 频数统计', fontsize=13, fontweight='bold')
axes[0].set_xlabel('点数')
axes[0].set_ylabel('出现次数')
axes[0].set_xticks(faces)
axes[0].legend()

# 右图：频率直方图 + 理论概率线
frequencies = counts / n_trials
axes[1].bar(faces, frequencies, color=colors, edgecolor='white', linewidth=1.5)
axes[1].axhline(y=1/6, color='black', linestyle='--', linewidth=1.5, label=f'理论概率: 1/6 ≈ {1/6:.3f}')
axes[1].set_title(f'掷骰子 {n_trials:,} 次 — 频率分布', fontsize=13, fontweight='bold')
axes[1].set_xlabel('点数')
axes[1].set_ylabel('频率')
axes[1].set_xticks(faces)
axes[1].set_ylim(0, 0.25)
axes[1].legend()

plt.tight_layout()
plt.show()

# 计算最大偏差
max_deviation = np.max(np.abs(frequencies - 1/6))
print(f"\n最大偏差: {max_deviation:.4f} ({max_deviation / (1/6) * 100:.1f}%)")

### 收敛性验证：随着试验次数增加，频率如何趋向理论值？

In [ ]:
rng = np.random.default_rng(seed=123)
n_max = 50_000
rolls = rng.integers(1, 7, size=n_max)

# 逐步计算频率
cumulative_freq = np.zeros((n_max, 6))
for i in range(6):
    cumulative_freq[:, i] = np.cumsum(rolls == (i + 1)) / np.arange(1, n_max + 1)

fig, ax = plt.subplots(figsize=(12, 5))
for i in range(6):
    ax.plot(range(1, n_max + 1), cumulative_freq[:, i], 
            linewidth=0.8, alpha=0.8, label=f'点数 {i+1}')
ax.axhline(y=1/6, color='black', linestyle='--', linewidth=1.5, label='理论值 1/6')
ax.set_xlabel('试验次数')
ax.set_ylabel('累计频率')
ax.set_title(f'频率收敛曲线 — 大数定律的直观展示', fontsize=13, fontweight='bold')
ax.legend(loc='center right', ncol=2)
ax.set_xlim(0, n_max)
ax.set_ylim(0.10, 0.22)
plt.tight_layout()
plt.show()

---

## 3.7 蒙特卡洛实战 ②：估算 π 值

**核心思想：** 在一个边长为 2 的正方形内随机撒点，统计落在内切圆内的比例。

```
        正方形面积 = 4
        内切圆面积 = π × 1² = π

        圆内点数 / 总点数 ≈ 圆面积 / 正方形面积 = π / 4

        ∴ π ≈ 4 × (圆内点数 / 总点数)
```

In [ ]:
rng = np.random.default_rng(seed=42)

# 参数设置
n_points = 10_000

# 在 [-1, 1] × [-1, 1] 正方形内随机撒点
x = rng.uniform(-1, 1, n_points)
y = rng.uniform(-1, 1, n_points)

# 计算每个点到原点的距离
distances = np.sqrt(x**2 + y**2)

# 距离 ≤ 1 的点在圆内
inside = distances <= 1
n_inside = np.sum(inside)

# 估算 π
pi_estimate = 4 * n_inside / n_points

print(f"总撒点数:   {n_points:,}")
print(f"圆内点数:   {n_inside:,}")
print(f"圆内比例:   {n_inside / n_points:.4f}")
print(f"π 估计值:   {pi_estimate:.6f}")
print(f"π 真实值:   {np.pi:.6f}")
print(f"相对误差:   {abs(pi_estimate - np.pi) / np.pi * 100:.4f}%")

In [ ]:
# 可视化撒点结果
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图：撒点图（只画前 2000 个点，避免太密）
n_show = min(2000, n_points)
axes[0].scatter(x[:n_show][inside[:n_show]], y[:n_show][inside[:n_show]], 
               c='steelblue', s=2, alpha=0.6, label='圆内点')
axes[0].scatter(x[:n_show][~inside[:n_show]], y[:n_show][~inside[:n_show]], 
               c='salmon', s=2, alpha=0.6, label='圆外点')

# 画圆和正方形
theta = np.linspace(0, 2*np.pi, 200)
axes[0].plot(np.cos(theta), np.sin(theta), 'black', linewidth=2, label='单位圆')
rect = plt.Rectangle((-1, -1), 2, 2, fill=False, edgecolor='black', linewidth=2, linestyle='--')
axes[0].add_patch(rect)
axes[0].set_aspect('equal')
axes[0].set_title(f'蒙特卡洛撒点 (显示前 {n_show} 个)', fontsize=13, fontweight='bold')
axes[0].set_xlim(-1.1, 1.1)
axes[0].set_ylim(-1.1, 1.1)
axes[0].legend(loc='upper right', markerscale=3)

# 右图：π 估计值随撒点数增加的收敛过程
cumulative_inside = np.cumsum(inside)
cumulative_pi = 4 * cumulative_inside / np.arange(1, n_points + 1)

axes[1].plot(range(1, n_points + 1), cumulative_pi, linewidth=0.8, color='steelblue')
axes[1].axhline(y=np.pi, color='red', linestyle='--', linewidth=1.5, label=f'π 真实值 = {np.pi:.6f}')
axes[1].fill_between(range(1, n_points + 1), cumulative_pi, np.pi, alpha=0.15, color='red')
axes[1].set_xlabel('撒点数量')
axes[1].set_ylabel('π 估计值')
axes[1].set_title(f'π 估计值的收敛过程 (最终: {pi_estimate:.4f})', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

### 不同样本量下的估算精度对比

蒙特卡洛方法的收敛速度是 O(1/√n)——想要多一位精确度，需要 100 倍的样本量。

In [ ]:
rng = np.random.default_rng(seed=2024)

sample_sizes = [100, 500, 1000, 5000, 10_000, 50_000, 100_000, 1_000_000]

print(f"{'样本量':>12}  {'π 估计值':>12}  {'误差':>12}  {'误差率':>10}")
print("-" * 55)

for n in sample_sizes:
    x = rng.uniform(-1, 1, n)
    y = rng.uniform(-1, 1, n)
    pi_est = 4 * np.sum(x**2 + y**2 <= 1) / n
    error = abs(pi_est - np.pi)
    error_pct = error / np.pi * 100
    print(f"{n:>12,}  {pi_est:>12.6f}  {error:>12.6f}  {error_pct:>9.4f}%")

---

## 📝 本讲小结

| 知识点 | 核心要点 |
|--------|---------|
| Generator API | 用 `default_rng(seed)` 替代 `np.random.seed()` |
| 种子 | 保证实验可复现，相同种子 = 相同序列 |
| `uniform` | 均匀分布，等概率随机 |
| `normal` | 正态分布，金融建模基石 |
| `integers` | 随机整数，掷骰子/抽样 |
| 蒙特卡洛 | 用大量随机试验近似复杂问题 |
| π 估算 | 撒点 → 比例 → 估算，精度随 √n 提升 |

---

**下一步：** 下一讲进入 Pandas，学习如何用 DataFrame 处理真实的金融数据。